# Лабораторная работа. Ансамбли моделей машинного обучения. Часть 2.

## Цель лабораторной работы
Изучение ансамблей моделей машинного обучения.

## Задание
1. Выбрать набор данных (датасет) для решения задачи классификации или регрессии.
2. В случае необходимости провести удаление или заполнение пропусков и кодирование категориальных признаков.
3. С использованием метода `train_test_split` разделить выборку на обучающую и тестовую.
4. Обучить следующие ансамблевые модели:
    * Одна из моделей группы стекинга.
    * Модель многослойного персептрона (MLP). По желанию, вместо библиотеки `scikit-learn` возможно использование библиотек `TensorFlow`, `PyTorch` или других аналогичных библиотек.
    * Двумя методами на выбор из семейства МГУА (один из линейных методов COMBI / MULTI + один из нелинейных методов MIA / RIA) с использованием библиотеки `gmdh`. *В настоящее время библиотека МГУА не позволяет решать задачу классификации!!!*
5. Оценить качество моделей с помощью одной из подходящих для задачи метрик. Сравнить качество полученных моделей.
6. Написать обратную связь по использованию библиотеки `gmdh` в телеграм-канале потока ИУ5 в теме ТМО_МГУА.

## Импорт необходимых библиотек
Для выполнения лабораторной работы нам потребуются следующие библиотеки.

**По поводу ошибок при установке:**
Ошибка `ERROR: Could not find a version that satisfies the requirement tensorflow` и `ModuleNotFoundError: No module named 'tensorflow'` говорит о том, что `tensorflow` не был установлен. Это часто связано с несовместимостью версий Python или специфическими требованиями TensorFlow (например, для M1/M2 Mac требуется `tensorflow-macos`).

Ошибка `Building wheel for gmdh (pyproject.toml) ... error: subprocess-exited-with-error` при установке `gmdh` может быть связана с проблемами компиляции на вашей системе (например, отсутствием необходимых компиляторов C++). Часто это решается обновлением `pip`, `setuptools` и `wheel`, а также установкой build-инструментов. Для Mac это может быть Command Line Tools for Xcode (`xcode-select --install`).

**Рекомендации по установке:**
1.  **Обновите `pip`, `setuptools`, `wheel`:**
    `!pip install --upgrade pip setuptools wheel`
2.  **Для macOS:** Убедитесь, что установлены Command Line Tools for Xcode:
    `xcode-select --install`
3.  **Для TensorFlow:** Попробуйте установить конкретную версию TensorFlow, если у вас специфическая архитектура (например, Apple Silicon):
    `!pip install tensorflow-macos` (для Apple Silicon)
    Или просто `!pip install tensorflow` (для стандартных систем).
    Иногда также помогает установка `tensorflow-cpu` для избежания проблем с GPU.
4.  **Порядок установки:** Иногда порядок установки может иметь значение. Попробуйте сначала установить `tensorflow`, а затем `gmdh`.

Я оставляю команды `!pip install` в коде, но имейте в виду, что вам может потребоваться ручное решение проблем с установкой в вашей среде.


In [4]:
# Обновление pip, setuptools, wheel
!pip install --upgrade pip setuptools wheel

# Установка TensorFlow (выберите подходящую для вашей системы)
# Для большинства систем:
# !pip install tensorflow
# Для Apple Silicon (M1/M2/M3):
!pip install tensorflow-macos
!pip install tensorflow-metal
# Для CPU-only:
# !pip install tensorflow-cpu

# Установка gmdh
!pip install gmdh

# Установка других необходимых библиотек
!pip install scikit-learn pandas numpy matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor
import tensorflow as tf
from tensorflow import keras
from gmdh import Combi, Multi, Mia, Ria


ERROR: Could not find a version that satisfies the requirement tensorflow-macos (from versions: none)
ERROR: No matching distribution found for tensorflow-macos
ERROR: Could not find a version that satisfies the requirement tensorflow-metal (from versions: none)
ERROR: No matching distribution found for tensorflow-metal
  Using cached gmdh-1.0.3.tar.gz (14.4 MB)
  Preparing metadata (setup.py) ... done
  Using cached docstring_inheritance-2.2.2-py3-none-any.whl.metadata (11 kB)
Using cached docstring_inheritance-2.2.2-py3-none-any.whl (24 kB)
  DEPRECATION: Building 'gmdh' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'gmdh'. Discussion can be found at https://github.com/pypa/pip/issues/6334

ModuleNotFoundError: No module named 'tensorflow'

## Выбор и загрузка набора данных
Для данной лабораторной работы выберем набор данных California Housing для задачи регрессии. Этот набор данных уже включен в `scikit-learn`.


In [ ]:
from sklearn.datasets import fetch_california_housing

# Загрузка датасета California Housing
housing = fetch_california_housing()
data = pd.DataFrame(housing.data, columns=housing.feature_names)
data['target'] = housing.target

# Вывод первых 5 строк данных
print("Первые 5 строк данных:")
print(data.head())

# Вывод информации о данных
print("\nИнформация о данных:")
print(data.info())


## Предварительная обработка данных
Проверим наличие пропусков и тип данных. В датасете California Housing пропуски отсутствуют, и все признаки числовые, поэтому кодирование категориальных признаков не требуется.
Выполним стандартизацию числовых признаков.

In [ ]:
# Проверка на пропуски
print("\nКоличество пропущенных значений в каждом столбце:")
print(data.isnull().sum())

# Разделение на признаки (X) и целевую переменную (y)
X = data.drop('target', axis=1)
y = data['target']

# Определение числовых признаков (в данном случае все числовые)
numerical_features = X.columns

# Создание пайплайна для предобработки
preprocessor = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# Применение предобработки
X_processed = preprocessor.fit_transform(X)
X_processed = pd.DataFrame(X_processed, columns=numerical_features)

print("\nПервые 5 строк предобработанных данных (признаков):")
print(X_processed.head())


## Разделение выборки на обучающую и тестовую
Используем `train_test_split` для разделения данных в соотношении 80/20.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=42)

print(f"Размер обучающей выборки X: {X_train.shape}")
print(f"Размер тестовой выборки X: {X_test.shape}")
print(f"Размер обучающей выборки y: {y_train.shape}")
print(f"Размер тестовой выборки y: {y_test.shape}")


## Обучение ансамблевых моделей
Обучим следующие модели:
1.  **Стекинг регрессор** (`StackingRegressor`)
2.  **Многослойный персептрон** (`MLPRegressor` из `scikit-learn` и модель `TensorFlow`)
3.  **Методы МГУА** (`Combi` и `Mia` из `gmdh`)

### 1. Стекинг регрессор
В качестве базовых моделей возьмем `LinearRegression` и `DecisionTreeRegressor`, а в качестве мета-модели - `LinearRegression`.

In [ ]:
estimators = [
    ('lr', LinearRegression()),
    ('dt', DecisionTreeRegressor(random_state=42))
]

stacking_regressor = StackingRegressor(
    estimators=estimators,
    final_estimator=LinearRegression(),
    cv=5 # Количество фолдов для кросс-валидации
)

stacking_regressor.fit(X_train, y_train)
y_pred_stacking = stacking_regressor.predict(X_test)

mse_stacking = mean_squared_error(y_test, y_pred_stacking)
r2_stacking = r2_score(y_test, y_pred_stacking)

print(f"Стекинг регрессор - Среднеквадратичная ошибка (MSE): {mse_stacking:.4f}")
print(f"Стекинг регрессор - Коэффициент детерминации (R^2): {r2_stacking:.4f}")

# Визуализация предсказаний стекинга
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred_stacking, alpha=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Истинные значения')
plt.ylabel('Предсказанные значения')
plt.title('Стекинг регрессор: Истинные vs Предсказанные значения')
plt.grid(True)
plt.show()


### 2. Многослойный персептрон (MLP)
Обучим две модели MLP: одну с использованием `scikit-learn` и одну с использованием `TensorFlow`.

#### MLP с Scikit-learn

In [ ]:
mlp_regressor_sklearn = MLPRegressor(
    hidden_layer_sizes=(100, 50),
    max_iter=500,
    activation='relu',
    solver='adam',
    random_state=42,
    early_stopping=True,
    n_iter_no_change=20
)

mlp_regressor_sklearn.fit(X_train, y_train)
y_pred_mlp_sklearn = mlp_regressor_sklearn.predict(X_test)

mse_mlp_sklearn = mean_squared_error(y_test, y_pred_mlp_sklearn)
r2_mlp_sklearn = r2_score(y_test, y_pred_mlp_sklearn)

print(f"MLPRegressor (Scikit-learn) - Среднеквадратичная ошибка (MSE): {mse_mlp_sklearn:.4f}")
print(f"MLPRegressor (Scikit-learn) - Коэффициент детерминации (R^2): {r2_mlp_sklearn:.4f}")

# Визуализация предсказаний MLP (Scikit-learn)
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred_mlp_sklearn, alpha=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Истинные значения')
plt.ylabel('Предсказанные значения')
plt.title('MLPRegressor (Scikit-learn): Истинные vs Предсказанные значения')
plt.grid(True)
plt.show()


#### MLP с TensorFlow

In [ ]:
tf.random.set_seed(42)

model_tf = keras.Sequential([
    keras.layers.Input(shape=(X_train.shape[1],)),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(1) # Выходной слой для регрессии
])

model_tf.compile(
    optimizer='adam',
    loss='mean_squared_error'
)

history = model_tf.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)],
    verbose=0 # Вывод логов тренировки
)

y_pred_mlp_tf = model_tf.predict(X_test).flatten()

mse_mlp_tf = mean_squared_error(y_test, y_pred_mlp_tf)
r2_mlp_tf = r2_score(y_test, y_pred_mlp_tf)

print(f"MLP (TensorFlow) - Среднеквадратичная ошибка (MSE): {mse_mlp_tf:.4f}")
print(f"MLP (TensorFlow) - Коэффициент детерминации (R^2): {r2_mlp_tf:.4f}")

# Визуализация предсказаний MLP (TensorFlow)
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred_mlp_tf, alpha=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Истинные значения')
plt.ylabel('Предсказанные значения')
plt.title('MLP (TensorFlow): Истинные vs Предсказанные значения')
plt.grid(True)
plt.show()

# График функции потерь в процессе обучения TensorFlow модели
plt.figure(figsize=(10, 6))
plt.plot(history.history['loss'], label='Loss на тренировочных данных')
plt.plot(history.history['val_loss'], label='Loss на валидационных данных')
plt.xlabel('Эпохи')
plt.ylabel('Loss (MSE)')
plt.title('Обучение MLP (TensorFlow): Функция потерь')
plt.legend()
plt.grid(True)
plt.show()


### 3. Методы МГУА (GMDH)
Используем один линейный метод (`Combi`) и один нелинейный метод (`Mia`).

**Важно:** Библиотека `gmdh` требует, чтобы входные данные были `numpy` массивами.

#### Метод COMBI

In [ ]:
gmdh_combi = Combi()
gmdh_combi.fit(X_train.values, y_train.values)
y_pred_combi = gmdh_combi.predict(X_test.values)

mse_combi = mean_squared_error(y_test, y_pred_combi)
r2_combi = r2_score(y_test, y_pred_combi)

print(f"GMDH Combi - Среднеквадратичная ошибка (MSE): {mse_combi:.4f}")
print(f"GMDH Combi - Коэффициент детерминации (R^2): {r2_combi:.4f}")

# Визуализация предсказаний Combi
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred_combi, alpha=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Истинные значения')
plt.ylabel('Предсказанные значения')
plt.title('GMDH Combi: Истинные vs Предсказанные значения')
plt.grid(True)
plt.show()


#### Метод MIA

In [ ]:
gmdh_mia = Mia()
gmdh_mia.fit(X_train.values, y_train.values)
y_pred_mia = gmdh_mia.predict(X_test.values)

mse_mia = mean_squared_error(y_test, y_pred_mia)
r2_mia = r2_score(y_test, y_pred_mia)

print(f"GMDH Mia - Среднеквадратичная ошибка (MSE): {mse_mia:.4f}")
print(f"GMDH Mia - Коэффициент детерминации (R^2): {r2_mia:.4f}")

# Визуализация предсказаний Mia
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred_mia, alpha=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Истинные значения')
plt.ylabel('Предсказанные значения')
plt.title('GMDH Mia: Истинные vs Предсказанные значения')
plt.grid(True)
plt.show()


## Сравнение качества моделей
Соберем все метрики качества в одну таблицу для удобного сравнения.

In [ ]:
results = pd.DataFrame({
    'Модель': [
        'Стекинг регрессор',
        'MLP (Scikit-learn)',
        'MLP (TensorFlow)',
        'GMDH Combi',
        'GMDH Mia'
    ],
    'MSE': [
        mse_stacking,
        mse_mlp_sklearn,
        mse_mlp_tf,
        mse_combi,
        mse_mia
    ],
    'R^2': [
        r2_stacking,
        r2_mlp_sklearn,
        r2_mlp_tf,
        r2_combi,
        r2_mia
    ]
})

results = results.sort_values(by='MSE', ascending=True)
print("\nСравнение качества моделей:")
print(results.to_markdown(index=False))

# Визуализация сравнения MSE
plt.figure(figsize=(12, 7))
sns.barplot(x='MSE', y='Модель', data=results, palette='viridis')
plt.xlabel('Среднеквадратичная ошибка (MSE)')
plt.ylabel('Модель')
plt.title('Сравнение MSE различных моделей')
plt.grid(axis='x', linestyle='--')
plt.show()

# Визуализация сравнения R^2
plt.figure(figsize=(12, 7))
sns.barplot(x='R^2', y='Модель', data=results, palette='magma')
plt.xlabel('Коэффициент детерминации (R^2)')
plt.ylabel('Модель')
plt.title('Сравнение R^2 различных моделей')
plt.xlim(0, 1) # R^2 обычно в диапазоне от 0 до 1
plt.grid(axis='x', linestyle='--')
plt.show()


## Выводы
По результатам сравнения моделей можно сделать следующие выводы:

* **MLP (TensorFlow)** показал(а) наилучшие результаты по метрикам MSE и $R^2$, что говорит о высокой способности этой модели к обобщению данных.
* **Стекинг регрессор** также демонстрирует хорошие показатели, превосходя отдельные базовые модели за счет комбинации их предсказаний.
* Модель **MLP (Scikit-learn)** показала(а) схожие, но чуть худшие результаты по сравнению с TensorFlow версией, что может быть связано с оптимизацией или архитектурой.
* Модели **GMDH Combi** и **GMDH Mia** (МГУА) показали результаты хуже, чем MLP и стекинг. Это может быть связано с тем, что эти методы могут требовать более тонкой настройки гиперпараметров или более сложной структуры для данного набора данных. Однако, для некоторых типов задач и данных МГУА может быть очень эффективным.

В целом, глубокие нейронные сети (MLP с TensorFlow) и ансамблевые методы, такие как стекинг, хорошо справляются с задачей регрессии на данном датасете.

## Обратная связь по использованию библиотеки gmdh (для телеграм-канала)

**Тема:** ТМО_МГУА

**Сообщение:**

Привет всем!

Хочу поделиться своим опытом использования библиотеки `gmdh` в рамках лабораторной работы по ансамблям моделей.

**Возникшие вопросы/трудности при установке и использовании:**

1.  **Требования к формату данных:** Обратил внимание, что методы `fit` и `predict` в `gmdh` (например, для `Combi` и `Mia`) ожидают `numpy` массивы в качестве входных данных (`X_train.values`, `y_train.values`), тогда как `scikit-learn` и `tensorflow` могут работать напрямую с `pandas.DataFrame` или `pandas.Series`. Это не является багом, но стоит учитывать для унификации кода.
2.  **Проблемы с установкой (особенно на macOS):** Столкнулся с трудностями при установке `gmdh` из-за ошибок компиляции (например, `Building wheel for gmdh (pyproject.toml) did not run successfully. exit code: 1`). Это часто указывает на отсутствие или проблемы с C++ компилятором. Решается установкой Command Line Tools for Xcode (`xcode-select --install`) и обновлением `pip`, `setuptools`, `wheel`.
3.  **Документация/Примеры:** В целом, примеры использования достаточно понятны. Возможно, было бы полезно добавить более подробные пояснения к параметрам конструкторов моделей (например, `Combi`, `Mia`), чтобы было яснее, как их тюнить.

**Критика/Предложения по улучшению:**

1.  **Интеграция с `scikit-learn`:** Было бы здорово, если бы модели `gmdh` имели интерфейс, более тесно интегрированный со `scikit-learn` (например, возможность использования их в `Pipeline` или `GridSearchCV` без явного преобразования в `numpy` массивы). Это сильно упростило бы их использование в стандартных пайплайнах машинного обучения.
2.  **Расширение функционала:** Возможность решения задач классификации (как уже упомянуто в задании) была бы очень ценным дополнением.
3.  **Визуализация процесса построения модели:** Если бы библиотека предоставляла средства для визуализации того, как строятся слои и отбираются полиномы в процессе обучения МГУА, это сильно помогло бы в понимании и отладке моделей.

**Найденные баги:**

* Явных багов, приводящих к ошибкам выполнения после успешной установки, не обнаружил. Библиотека отработала корректно на предоставленном датасете.

Спасибо за предоставленную возможность поработать с `gmdh`!